# Report for practical lesson 02: Convolutional Neural Network

In [5]:
import pandas as pd
import os, sys
import numpy as np
project_root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root_path not in sys.path:
    sys.path.append(project_root_path)

from sklearn.model_selection import train_test_split
import torch

import matplotlib.pyplot as plt
import torchvision.transforms as transforms

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter # Sử dụng TensorBoard logger
from sklearn.metrics import classification_report
import numpy as np
import os
import time

This box imports customized modules

In [6]:
from src.data.load import load_data
from src.train_model import train_model

Crucial variables

In [7]:
DATA_PATH = r"c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\lesson-02\dataset"

## Data Preprocessing

### Work 01: Load dataset

In [8]:
original_train, original_val, original_test, classes = load_data(
    data_root=DATA_PATH,
    num_workers=0, pin_memory=False,
    batch_size=32
)

### Work 02: Turn dataset into tensor format

In [9]:
def create_tensor_loaders(train_loader, val_loader, test_loader):
    """
    Create tensor transforms and corresponding DataLoaders for train, validation, and test datasets.

    Args:
        train_loader (torch.utils.data.DataLoader): DataLoader for the training dataset.
        val_loader (torch.utils.data.DataLoader): DataLoader for the validation dataset.
        test_loader (torch.utils.data.DataLoader): DataLoader for the test dataset.

    Returns:
        tuple: (train_tensor_loader, val_tensor_loader, test_tensor_loader)
            - train_tensor_loader: DataLoader with tensor transform applied to training data.
            - val_tensor_loader: DataLoader with tensor transform applied to validation data.
            - test_tensor_loader: DataLoader with tensor transform applied to test data.
    """

    train_tensor_loader = torch.utils.data.DataLoader(
        train_loader.dataset,
        batch_size=train_loader.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )

    val_tensor_loader = torch.utils.data.DataLoader(
        val_loader.dataset,
        batch_size=val_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    test_tensor_loader = torch.utils.data.DataLoader(
        test_loader.dataset,
        batch_size=test_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    return train_tensor_loader, val_tensor_loader, test_tensor_loader

In [10]:
train_tensor_loader, val_tensor_loader, test_tensor_loader = create_tensor_loaders(
    original_train, original_val, original_test
)

images, labels = next(iter(train_tensor_loader))
print("Tensor shape:", images.shape)
print("Data type:", images.dtype)

train_tensor_loader

Tensor shape: torch.Size([32, 3, 224, 224])
Data type: torch.float32


### Work 02: Transform data into grayscale

In [11]:
def to_grayscale(train_loader, val_loader, test_loader):
    """
    Convert RGB images to grayscale for each data loader
    
    Args:
        train_loader: DataLoader with RGB images
        val_loader: DataLoader with RGB images
        test_loader: DataLoader with RGB images
        
    Returns:
        Three DataLoaders containing grayscale images
    """
    grayscale_transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1)
    ])
    
    # Create new dataloaders with grayscale transform
    train_gray = torch.utils.data.DataLoader(
        train_loader.dataset,
        batch_size=train_loader.batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )
    
    val_gray = torch.utils.data.DataLoader(
        val_loader.dataset,
        batch_size=val_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    test_gray = torch.utils.data.DataLoader(
        test_loader.dataset,
        batch_size=test_loader.batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )
    
    return train_gray, val_gray, test_gray

In [12]:
train, val, test = to_grayscale(
    train_tensor_loader, 
    val_tensor_loader, 
    test_tensor_loader
)

In [13]:
images, labels = next(iter(train))
print("Tensor shape:", images.shape)
print("Data type:", images.dtype)

train_tensor_loader

Tensor shape: torch.Size([32, 3, 224, 224])
Data type: torch.float32


## Session 01

### Worrk 01: Build Neural Network

The current model was establish via Python code (`lesson-02\networks\model01.py`). The bellow code snippet is to load model into this notebook.

In [14]:
from src.networks.model01 import model01

In [15]:
print("Number of classes:", len(classes))
instance01 = model01(num_classes=len(classes))
print("Instance 01 created successfully.")

Number of classes: 21
Instance 01 created successfully.


To print model description, run this.

In [16]:
print(instance01)

model01(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (pool1): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool2): AvgPool2d(kernel_size=2, stride=2, padding=0)
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=21, bias=True)
)


### Work 02: Train model

In [17]:
from src.train_model import train_model

In [18]:
def preprocessing_fn(batch: torch.Tensor) -> torch.Tensor:
    # Convert from [B, 3, 224, 224] -> [B, 1, 28, 28]
    gray = batch.mean(dim=1, keepdim=True)  # RGB to grayscale
    resized = torch.nn.functional.interpolate(gray, size=(28, 28), mode='bilinear', align_corners=False)
    return resized

In [19]:
dev = print('cuda' if torch.cuda.is_available() else 'cpu')

cuda


In [20]:
trained_model, history, best_ckpt_path, test_metrics = train_model(
    train_loader=train,
    val_loader=val,
    model=instance01,  
    epochs=10,
    lr=0.005,
    preprocessing_fn=preprocessing_fn,
    device=dev,
    model_args=None 
)

Epoch 1/10 - train:   2%|▏         | 5/283 [00:03<02:52,  1.61batch/s, loss=3.02]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

## Session 02

### Work 01: Neural Network Design

In [21]:
from src.networks.model02 import GoogLeNet

In [23]:
instance02 = GoogLeNet(num_classes=len(classes))
print(instance02)

GoogLeNet(
  (stem): Sequential(
    (0): ConvBlock(
      (conv): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (1): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (2): ConvBlock(
      (conv): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (3): ConvBlock(
      (conv): Conv2d(64, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(192, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
    )
    (4): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (inception_3a): Inception(
    (branch1): ConvBlock(
      (conv): Conv2d(192, 

### Work 02: Train model

In [27]:
device_for_instance02 = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device_for_instance02}')

Using device: cuda


In [42]:
def train_instance02(
    # --- Nhóm Cốt lõi ---
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    
    # --- Nhóm Cấu hình ---
    num_epochs: int,
    device: torch.device,
    
    # --- Nhóm MLOps ---
    checkpoint_dir: str,
    logger: SummaryWriter
):
    """
    Hàm train chuyên biệt cho GoogLeNet (InceptionV1).
    Xử lý 3 output khi training và 1 output khi validation.
    """
    
    # Tạo thư mục checkpoint nếu chưa có
    if not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
        
    best_val_loss = float('inf')
    
    print(f"Bắt đầu training trên thiết bị: {device}")
    
    for epoch in range(num_epochs):
        
        # ==========================
        #      PHA TRAINING
        # ==========================
        model.train() # QUAN TRỌNG: Bật chế độ train
                      # (GoogLeNet sẽ trả về 3 output)
        
        running_train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Xóa gradient
            optimizer.zero_grad()
            
            # --- PHẦN LOGIC ĐẶC BIỆT CỦA GOOGLENET ---
            # Forward pass - nhận 3 output
            main_output, aux1_output, aux2_output = model(inputs)
            
            # Tính loss cho cả 3
            loss_main = criterion(main_output, labels)
            loss_aux1 = criterion(aux1_output, labels)
            loss_aux2 = criterion(aux2_output, labels)
            
            # Tổng loss theo trọng số của paper
            loss = loss_main + 0.3 * loss_aux1 + 0.3 * loss_aux2
            # --- KẾT THÚC PHẦN LOGIC ĐẶC BIỆT ---
            
            # Backward
            loss.backward()
            
            # Optimize
            optimizer.step()
            
            running_train_loss += loss.item()
            
        avg_train_loss = running_train_loss / len(train_loader)
        
        
        # ==========================
        #     PHA VALIDATION
        # ==========================
        model.eval() # QUAN TRỌNG: Bật chế độ eval
                     # (GoogLeNet sẽ chỉ trả về 1 output chính)
        
        running_val_loss = 0.0
        all_preds = []
        all_labels = []
        
        with torch.no_grad(): # Không cần tính gradient
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                
                # Forward pass - chỉ nhận 1 output
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item()
                
                # Lấy dự đoán để tính classification report
                _, preds = torch.max(outputs, 1)
                all_preds.append(preds.cpu().numpy())
                all_labels.append(labels.cpu().numpy())
                
        avg_val_loss = running_val_loss / len(val_loader)
        
        # Nối các kết quả từ các batch
        all_preds = np.concatenate(all_preds)
        all_labels = np.concatenate(all_labels)
        
        # ==========================
        #     LOGGING & CHECKPOINT
        # ==========================
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss:   {avg_val_loss:.4f}")
        
        # Log ra TensorBoard
        logger.add_scalar('Loss/train', avg_train_loss, epoch)
        logger.add_scalar('Loss/validation', avg_val_loss, epoch)
        
        # In classification report (precision, recall, f1)
        # Lấy tên class từ loader (nếu có) hoặc tạo tên giả
        try:
            class_names = val_loader.dataset.classes
        except:
            class_names = [f'Class {i}' for i in range(len(np.unique(all_labels)))]
            
        report = classification_report(
            all_labels, 
            all_preds, 
            target_names=class_names, 
            zero_division=0,
            digits=4
        )
        print("--- Báo cáo đánh giá (Precision, Recall, F1) ---")
        print(report)
        
        # Log F1-score (macro avg) vào TensorBoard
        # (Bạn có thể parse 'report' hoặc tính riêng, ở đây tôi log F1-score từ report)
        report_dict = classification_report(all_labels, all_preds, zero_division=0, output_dict=True)
        logger.add_scalar('F1-Score/macro_avg', report_dict['macro avg']['f1-score'], epoch)
        logger.add_scalar('Accuracy/validation', report_dict['accuracy'], epoch)

        # Lưu checkpoint của model tốt nhất
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            save_path = os.path.join(checkpoint_dir, "best_model.pth")
            torch.save(model.state_dict(), save_path)
            print(f"==> Model tốt nhất đã được lưu tại: {save_path}")
            
    print("\nTraining hoàn tất.")
    logger.close()

In [35]:
for images, labels in train:
    print("Number of batches:", len(train))
    print("Number of classes:", len(classes))
    print("Tensor shape:", images.shape)
    print("Data type:", images.dtype)
    break

Number of batches: 283
Number of classes: 21
Tensor shape: torch.Size([32, 3, 224, 224])
Data type: torch.float32


In [37]:
NUM_CLASSES = 21
NUM_EPOCHS = 15
LEARNING_RATE = 0.001
MODEL_NAME = "GoogLeNet"
CHECKPOINT_DIR = f"./checkpoints/{MODEL_NAME}"
LOG_DIR = f"./logs/{MODEL_NAME}_{int(time.time())}"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [40]:
print(f"Đang khởi tạo các thành phần cho {MODEL_NAME}...")
model = GoogLeNet(num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
logger = SummaryWriter(log_dir=LOG_DIR)

print(f"Model: {MODEL_NAME}, Num Classes: {NUM_CLASSES}")
print(f"Device: {device}")
print(f"Checkpoints sẽ lưu tại: {CHECKPOINT_DIR}")
print(f"Logs TensorBoard sẽ lưu tại: {LOG_DIR}")

Đang khởi tạo các thành phần cho GoogLeNet...
Model: GoogLeNet, Num Classes: 21
Device: cuda
Checkpoints sẽ lưu tại: ./checkpoints/GoogLeNet
Logs TensorBoard sẽ lưu tại: ./logs/GoogLeNet_1762075265


In [41]:
train_instance02(
    model=model,
    train_loader=train, # <-- Biến 'train' của bạn
    val_loader=val,     # <-- Biến 'val' của bạn
    criterion=criterion,
    optimizer=optimizer,
    num_epochs=NUM_EPOCHS,
    device=device,
    checkpoint_dir=CHECKPOINT_DIR,
    logger=logger
)

Bắt đầu training trên thiết bị: cuda


c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 1/15
  Train Loss: 4.3777
  Val Loss:   3.1371
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000        41
     Class 1     0.1500    0.3956    0.2175        91
     Class 2     0.0000    0.0000    0.0000        30
     Class 3     0.1761    0.5435    0.2660        46
     Class 4     0.0000    0.0000    0.0000        47
     Class 5     0.0000    0.0000    0.0000        21
     Class 6     0.1429    0.0235    0.0404        85
     Class 7     0.0000    0.0000    0.0000        28
     Class 8     0.0000    0.0000    0.0000        47
     Class 9     0.2500    0.0580    0.0941        69
    Class 10     0.0250    0.0233    0.0241        43
    Class 11     0.0476    0.0385    0.0426        26
    Class 12     0.4643    0.3514    0.4000        37
    Class 13     0.1818    0.7143    0.2899        28
    Class 14     0.1382    0.5758    0.2229        66
    Class 15     0.0000    0.000

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 2/15
  Train Loss: 4.0647
  Val Loss:   2.3989
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.0000    0.0000    0.0000        41
     Class 1     0.4032    0.2747    0.3268        91
     Class 2     0.1034    0.3000    0.1538        30
     Class 3     0.2903    0.7826    0.4235        46
     Class 4     0.1282    0.1064    0.1163        47
     Class 5     0.0000    0.0000    0.0000        21
     Class 6     0.2609    0.2824    0.2712        85
     Class 7     0.0000    0.0000    0.0000        28
     Class 8     0.3000    0.1277    0.1791        47
     Class 9     0.4609    0.7681    0.5761        69
    Class 10     0.1667    0.0233    0.0408        43
    Class 11     0.0000    0.0000    0.0000        26
    Class 12     0.2951    0.4865    0.3673        37
    Class 13     1.0000    0.3214    0.4865        28
    Class 14     0.3153    0.5303    0.3955        66
    Class 15     0.2568    0.365

...


Epoch 13/15
  Train Loss: 2.8557
  Val Loss:   1.7363
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.6000    0.0732    0.1304        41
     Class 1     0.4842    0.5055    0.4946        91
     Class 2     0.3000    0.5000    0.3750        30
     Class 3     0.7381    0.6739    0.7045        46
     Class 4     0.4151    0.4681    0.4400        47
     Class 5     0.2667    0.1905    0.2222        21
     Class 6     0.4066    0.4353    0.4205        85
     Class 7     0.5714    0.1429    0.2286        28
     Class 8     0.2662    0.7872    0.3978        47
     Class 9     0.6111    0.7971    0.6918        69
    Class 10     0.2812    0.2093    0.2400        43
    Class 11     0.0000    0.0000    0.0000        26
    Class 12     0.6667    0.6486    0.6575        37
    Class 13     0.6857    0.8571    0.7619        28
    Class 14     0.6957    0.4848    0.5714        66
    Class 15     0.7576    0.48

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 14/15
  Train Loss: 2.7610
  Val Loss:   1.7265
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.7500    0.2195    0.3396        41
     Class 1     0.6471    0.4835    0.5535        91
     Class 2     0.3077    0.2667    0.2857        30
     Class 3     0.8140    0.7609    0.7865        46
     Class 4     0.4062    0.2766    0.3291        47
     Class 5     0.3077    0.1905    0.2353        21
     Class 6     0.3727    0.4824    0.4205        85
     Class 7     0.8571    0.2143    0.3429        28
     Class 8     0.3298    0.6596    0.4397        47
     Class 9     0.7286    0.7391    0.7338        69
    Class 10     0.3125    0.4651    0.3738        43
    Class 11     0.5714    0.1538    0.2424        26
    Class 12     0.4894    0.6216    0.5476        37
    Class 13     1.0000    0.6786    0.8085        28
    Class 14     0.5625    0.2727    0.3673        66
    Class 15     0.6800    0.32

c:\Users\VICTUS\Documents\developer\UIT_year3_sem1\ds201-DL-practicalLesson\venv\lib\site-packages\PIL\Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



Epoch 15/15
  Train Loss: 2.6978
  Val Loss:   1.6472
--- Báo cáo đánh giá (Precision, Recall, F1) ---
              precision    recall  f1-score   support

     Class 0     0.5000    0.4390    0.4675        41
     Class 1     0.6500    0.4286    0.5166        91
     Class 2     0.3548    0.3667    0.3607        30
     Class 3     0.7778    0.7609    0.7692        46
     Class 4     0.4340    0.4894    0.4600        47
     Class 5     0.4545    0.2381    0.3125        21
     Class 6     0.4940    0.4824    0.4881        85
     Class 7     0.3810    0.2857    0.3265        28
     Class 8     0.3768    0.5532    0.4483        47
     Class 9     0.3721    0.9275    0.5311        69
    Class 10     0.4000    0.4186    0.4091        43
    Class 11     0.5833    0.2692    0.3684        26
    Class 12     0.5294    0.4865    0.5070        37
    Class 13     0.7857    0.7857    0.7857        28
    Class 14     0.6800    0.5152    0.5862        66
    Class 15     0.6207    0.34